# 📊 JalanCerdas AI - Dataset Exploration

Explore and analyze the pothole detection dataset:
- Class distribution
- Image size distribution
- Annotation statistics
- Sample visualizations

## Cell 1: Setup & Load Dataset

In [ ]:
!pip install -q matplotlib Pillow pyyaml tqdm

import os
import json
from pathlib import Path
from collections import Counter, defaultdict
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import yaml
import random

# Dataset path
DATASET_PATH = Path("../dataset")

print(f"Dataset directory: {DATASET_PATH.absolute()}")
print(f"Exists: {DATASET_PATH.exists()}")

In [ ]:
# Load config
data_yaml = DATASET_PATH / "data.yaml"

if data_yaml.exists():
    with open(data_yaml) as f:
        config = yaml.safe_load(f)
    print(f"Classes: {config['nc']} - {config['names']}")
else:
    print("⚠️  data.yaml not found. Using default pothole config.")
    config = {'nc': 1, 'names': {0: 'pothole'}, 'train': 'train/images', 'val': 'val/images', 'test': 'test/images'}

# Scan dataset
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
dataset_info = {}

for split in ['train', 'val', 'test']:
    img_dir = DATASET_PATH / split / 'images'
    lbl_dir = DATASET_PATH / split / 'labels'
    
    images = []
    if img_dir.exists():
        for f in img_dir.iterdir():
            if f.suffix.lower() in IMAGE_EXTS:
                images.append(f)
    
    labels = []
    if lbl_dir.exists():
        labels = list(lbl_dir.glob('*.txt'))
    
    dataset_info[split] = {
        'images': sorted(images),
        'labels': sorted(labels),
    }
    print(f"{split}: {len(images)} images, {len(labels)} labels")

total_images = sum(len(v['images']) for v in dataset_info.values())
total_labels = sum(len(v['labels']) for v in dataset_info.values())
print(f"\nTotal: {total_images} images, {total_labels} label files")

## Cell 2: Class Distribution

In [ ]:
# Count classes across all labels
class_counts = Counter()
annotations_per_image = []
total_annotations = 0

for split_data in dataset_info.values():
    for lbl_file in split_data['labels']:
        with open(lbl_file) as f:
            lines = f.readlines()
            n_annots = len([l for l in lines if l.strip()])
            annotations_per_image.append(n_annots)
            for line in lines:
                parts = line.strip().split()
                if parts:
                    cls_id = int(parts[0])
                    class_counts[cls_id] += 1
                    total_annotations += 1

print(f"Total annotations: {total_annotations}")
print(f"\nClass distribution:")
for cls_id, count in sorted(class_counts.items()):
    name = config['names'].get(cls_id, f'class_{cls_id}')
    pct = count / total_annotations * 100 if total_annotations > 0 else 0
    print(f"  {name} (id={cls_id}): {count} annotations ({pct:.1f}%)")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Class distribution bar chart
names = [config['names'].get(i, f'c{i}') for i in sorted(class_counts.keys())]
counts = [class_counts[i] for i in sorted(class_counts.keys())]
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7'][:len(names)]

axes[0].bar(names, counts, color=colors)
axes[0].set_title('Class Distribution')
axes[0].set_ylabel('Number of Annotations')
for i, (n, c) in enumerate(zip(names, counts)):
    axes[0].text(i, c + max(counts)*0.02, str(c), ha='center', fontweight='bold')

# Split distribution
split_counts = [len(dataset_info[s]['images']) for s in ['train', 'val', 'test']]
axes[1].bar(['train', 'val', 'test'], split_counts, color=['#2ecc71', '#f39c12', '#e74c3c'])
axes[1].set_title('Images per Split')
axes[1].set_ylabel('Number of Images')
for i, c in enumerate(split_counts):
    axes[1].text(i, c + max(split_counts)*0.02, str(c), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

## Cell 3: Image Size Distribution

In [ ]:
# Analyze image dimensions
widths = []
heights = []
aspect_ratios = []

print("Analyzing image sizes...")
for split_data in dataset_info.values():
    for img_path in split_data['images']:
        try:
            with Image.open(img_path) as img:
                w, h = img.size
                widths.append(w)
                heights.append(h)
                aspect_ratios.append(w / h)
        except Exception as e:
            print(f"  Error reading {img_path}: {e}")

if widths:
    print(f"\nImage dimensions ({len(widths)} images):")
    print(f"  Width:  min={min(widths)}, max={max(widths)}, mean={np.mean(widths):.0f}, median={np.median(widths):.0f}")
    print(f"  Height: min={min(heights)}, max={max(heights)}, mean={np.mean(heights):.0f}, median={np.median(heights):.0f}")
    print(f"  Aspect ratio: min={min(aspect_ratios):.2f}, max={max(aspect_ratios):.2f}, mean={np.mean(aspect_ratios):.2f}")
    
    # Calculate total dataset size
    total_pixels = sum(w * h for w, h in zip(widths, heights))
    total_mb = total_pixels * 3 / (1024 * 1024)  # RGB
    print(f"  Total raw RGB data: ~{total_mb:.1f} MB")
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    axes[0].hist(widths, bins=30, color='#3498db', alpha=0.7, edgecolor='black')
    axes[0].set_title('Width Distribution')
    axes[0].set_xlabel('Width (px)')
    axes[0].set_ylabel('Count')
    axes[0].axvline(x=640, color='red', linestyle='--', label='YOLO input (640)')
    axes[0].legend()
    
    axes[1].hist(heights, bins=30, color='#2ecc71', alpha=0.7, edgecolor='black')
    axes[1].set_title('Height Distribution')
    axes[1].set_xlabel('Height (px)')
    axes[1].set_ylabel('Count')
    axes[1].axvline(x=640, color='red', linestyle='--', label='YOLO input (640)')
    axes[1].legend()
    
    axes[2].hist(aspect_ratios, bins=30, color='#9b59b6', alpha=0.7, edgecolor='black')
    axes[2].set_title('Aspect Ratio Distribution')
    axes[2].set_xlabel('Width / Height')
    axes[2].set_ylabel('Count')
    axes[2].axvline(x=1.0, color='red', linestyle='--', label='Square (1.0)')
    axes[2].legend()
    
    plt.tight_layout()
    plt.show()
else:
    print("No images found to analyze.")

## Cell 4: Annotation Statistics

In [ ]:
# Analyze bounding box properties
bbox_widths = []  # Normalized
bbox_heights = []
bbox_areas = []
bbox_centers_x = []
bbox_centers_y = []
annotations_per_img = []

for split_data in dataset_info.values():
    for lbl_file in split_data['labels']:
        with open(lbl_file) as f:
            lines = [l.strip() for l in f if l.strip()]
            annotations_per_img.append(len(lines))
            
            for line in lines:
                parts = line.split()
                if len(parts) >= 5:
                    cx, cy, bw, bh = [float(x) for x in parts[1:5]]
                    bbox_centers_x.append(cx)
                    bbox_centers_y.append(cy)
                    bbox_widths.append(bw)
                    bbox_heights.append(bh)
                    bbox_areas.append(bw * bh)

if bbox_widths:
    print(f"Annotation statistics ({sum(annotations_per_img)} total annotations):")
    print(f"  Annotations per image: min={min(annotations_per_img)}, max={max(annotations_per_img)}, "
          f"mean={np.mean(annotations_per_img):.1f}, median={np.median(annotations_per_img):.0f}")
    print(f"  BBox width (normalized):  min={min(bbox_widths):.4f}, max={max(bbox_widths):.4f}, mean={np.mean(bbox_widths):.4f}")
    print(f"  BBox height (normalized): min={min(bbox_heights):.4f}, max={max(bbox_heights):.4f}, mean={np.mean(bbox_heights):.4f}")
    print(f"  BBox area (normalized):   min={min(bbox_areas):.6f}, max={max(bbox_areas):.4f}, mean={np.mean(bbox_areas):.4f}")
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    
    # Annotations per image
    axes[0,0].hist(annotations_per_img, bins=range(0, max(annotations_per_img)+2), color='#e74c3c', alpha=0.7, edgecolor='black')
    axes[0,0].set_title('Annotations per Image')
    axes[0,0].set_xlabel('Number of Annotations')
    axes[0,0].set_ylabel('Count')
    
    # BBox width
    axes[0,1].hist(bbox_widths, bins=30, color='#3498db', alpha=0.7, edgecolor='black')
    axes[0,1].set_title('BBox Width Distribution (Normalized)')
    axes[0,1].set_xlabel('Width')
    
    # BBox height
    axes[0,2].hist(bbox_heights, bins=30, color='#2ecc71', alpha=0.7, edgecolor='black')
    axes[0,2].set_title('BBox Height Distribution (Normalized)')
    axes[0,2].set_xlabel('Height')
    
    # BBox area
    axes[1,0].hist(bbox_areas, bins=30, color='#f39c12', alpha=0.7, edgecolor='black')
    axes[1,0].set_title('BBox Area Distribution')
    axes[1,0].set_xlabel('Area (normalized)')
    
    # Center distribution (where potholes appear)
    axes[1,1].scatter(bbox_centers_x, bbox_centers_y, alpha=0.1, s=5, c='#9b59b6')
    axes[1,1].set_title('BBox Center Positions')
    axes[1,1].set_xlabel('Center X')
    axes[1,1].set_ylabel('Center Y')
    axes[1,1].set_xlim(0, 1)
    axes[1,1].set_ylim(1, 0)  # Invert Y (image coords)
    axes[1,1].set_aspect('equal')
    
    # Width vs Height scatter
    axes[1,2].scatter(bbox_widths, bbox_heights, alpha=0.1, s=5, c='#1abc9c')
    axes[1,2].set_title('BBox Width vs Height')
    axes[1,2].set_xlabel('Width')
    axes[1,2].set_ylabel('Height')
    
    plt.tight_layout()
    plt.show()
else:
    print("No annotations found to analyze.")

## Cell 5: Sample Visualizations

In [ ]:
# Show samples from each split with annotations
CLASS_COLORS = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7']

fig, axes = plt.subplots(3, 3, figsize=(15, 15))

row = 0
for split in ['train', 'val', 'test']:
    images = dataset_info[split]['images']
    if not images:
        axes[row, 0].text(0.5, 0.5, f'No {split} images', ha='center', va='center')
        axes[row, 0].set_title(f'{split} (empty)')
        axes[row, 1].axis('off')
        axes[row, 2].axis('off')
        row += 1
        continue
    
    samples = random.sample(images, min(3, len(images)))
    for col, img_path in enumerate(samples):
        img = Image.open(img_path)
        draw = ImageDraw.Draw(img)
        w, h = img.size
        
        # Load and draw labels
        lbl_path = dataset_info[split]['labels'][0].parent / (img_path.stem + '.txt')
        n_boxes = 0
        if not lbl_path.exists():
            lbl_path = DATASET_PATH / split / 'labels' / (img_path.stem + '.txt')
        
        if lbl_path.exists():
            with open(lbl_path) as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) >= 5:
                        cls_id = int(parts[0])
                        cx, cy, bw, bh = [float(x) for x in parts[1:5]]
                        x1 = int((cx - bw/2) * w)
                        y1 = int((cy - bh/2) * h)
                        x2 = int((cx + bw/2) * w)
                        y2 = int((cy + bh/2) * h)
                        color = CLASS_COLORS[cls_id % len(CLASS_COLORS)]
                        draw.rectangle([x1, y1, x2, y2], outline=color, width=3)
                        draw.text((x1, max(0, y1-15)), config['names'].get(cls_id, f'c{cls_id}'), fill=color)
                        n_boxes += 1
        
        axes[row, col].imshow(img)
        axes[row, col].set_title(f"{split}/{img_path.name}\n({n_boxes} potholes)", fontsize=9)
        axes[row, col].axis('off')
    
    row += 1

plt.suptitle('Dataset Samples by Split', fontsize=14)
plt.tight_layout()
plt.show()

## Cell 6: Size Distribution by BBox Area

In [ ]:
# Categorize potholes by size
if bbox_areas:
    small = sum(1 for a in bbox_areas if a < 0.01)      # < 1% of image
    medium = sum(1 for a in bbox_areas if 0.01 <= a < 0.05)  # 1-5%
    large = sum(1 for a in bbox_areas if 0.05 <= a < 0.15)   # 5-15%
    xlarge = sum(1 for a in bbox_areas if a >= 0.15)          # > 15%
    
    total = len(bbox_areas)
    print("Pothole Size Distribution:")
    print(f"  Small  (<1%):    {small:4d} ({small/total*100:.1f}%)")
    print(f"  Medium (1-5%):   {medium:4d} ({medium/total*100:.1f}%)")
    print(f"  Large  (5-15%):  {large:4d} ({large/total*100:.1f}%)")
    print(f"  XLarge (>15%):   {xlarge:4d} ({xlarge/total*100:.1f}%)")
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # Pie chart
    labels = ['Small', 'Medium', 'Large', 'XLarge']
    sizes = [small, medium, large, xlarge]
    colors = ['#3498db', '#2ecc71', '#f39c12', '#e74c3c']
    # Filter zero values
    non_zero = [(l, s, c) for l, s, c in zip(labels, sizes, colors) if s > 0]
    if non_zero:
        labels_nz, sizes_nz, colors_nz = zip(*non_zero)
        axes[0].pie(sizes_nz, labels=labels_nz, colors=colors_nz, autopct='%1.1f%%', startangle=90)
    axes[0].set_title('Pothole Size Distribution')
    
    # Histogram of areas
    axes[1].hist(bbox_areas, bins=50, color='#9b59b6', alpha=0.7, edgecolor='black')
    axes[1].set_title('BBox Area Distribution')
    axes[1].set_xlabel('Area (normalized)')
    axes[1].set_ylabel('Count')
    axes[1].axvline(x=0.01, color='blue', linestyle='--', alpha=0.5, label='Small/Medium')
    axes[1].axvline(x=0.05, color='green', linestyle='--', alpha=0.5, label='Medium/Large')
    axes[1].axvline(x=0.15, color='red', linestyle='--', alpha=0.5, label='Large/XL')
    axes[1].legend()
    
    plt.tight_layout()
    plt.show()
else:
    print("No bounding box data to analyze.")

## Summary

**Key observations:**
- Dataset contains pothole bounding box annotations in YOLO format
- Images vary in size — YOLO will resize to 640x640 during training
- Class distribution shows single-class (pothole) detection
- BBox center positions reveal where potholes typically appear (road center)
- Size analysis helps understand detection difficulty (small = harder)

**Recommendations:**
- Balance train/val/test splits
- Ensure sufficient small pothole samples
- Consider augmentation for small objects (mosaic, copy-paste)